# LearnMateAI — Qwen 2.5 LoRA Fine-Tuning

Colab-ready notebook for **LoRA/PEFT** fine-tuning (not full fine-tuning) on Stage 3 dataset output.

## Golden path (avoids every error hit while building this)

1. **Runtime → Disconnect and delete runtime** (start clean if you tried installs before)
2. **Runtime → Change runtime type → GPU → T4**
3. Run cell **"0 — Install dependencies"** once
4. **Runtime → Restart session** (required — do not skip)
5. Run cell **"0b — Verify environment"** — every line should print a version, not `IMPORT FAILED`
6. Run the rest top to bottom: CONFIG → dataset → tokenizer/model → LoRA → auto-tune → train → save → run-record → download

## Dataset for this run

Upload `train.jsonl` and `val.jsonl` from **`01_dataset_pipeline/processed_v01/`** (`lm-legal-v0.1` — 1,590 train / 339 val). Do *not* use `02_finetuning/sample_data/` (synthetic smoke-v1) and do *not* use `processed/` (smoke-v2, also synthetic). Keep `test.jsonl` and `test_strict.jsonl` on your machine for evaluation after training — they are not uploaded to Colab.

## T4 free-tier sizing

Tuned to stay well inside a free T4's 16 GB with room to grow the corpus:

- **4-bit QLoRA (nf4 + double quant)** on Qwen2.5-1.5B — base weights are ~1.1 GB
- **SDPA attention** — FlashAttention-2 needs Ampere+, so SDPA is the best a T4 can do
- **Gradient checkpointing** with `use_reentrant=False`
- **`prediction_loss_only`** — Qwen's ~152k vocab makes gathered eval logits the single biggest OOM risk, and only `eval_loss` is actually needed
- **`paged_adamw_8bit`** — optimizer state in 8-bit, paged to host RAM on a spike
- **Auto-tuned `max_seq_length`** — measured from real token lengths (~430 max) rather than the old hardcoded 1024, which padded every batch to more than twice what the data needs

Cell **4b** prints peak-VRAM headroom after training. If it drops under 1.5 GB, lower `per_device_train_batch_size` before enlarging the corpus.

**Never** run `pip install --force-reinstall` on `torch`, `torchvision`, `pillow`, or `numpy` in Colab — that is what caused the `torchvision::nms`, `PIL._typing._Ink`, and `numpy.dtype size changed` errors during development. Colab's preinstalled build of those four packages already matches its GPU driver/CUDA; leave them alone and only add the Hugging Face training libraries.

**Budget note (~USD 45/mo ops):** Prefer `Qwen/Qwen2.5-1.5B-Instruct` + QLoRA on free/cheap Colab. Larger bases only if a sponsored GPU is available.

**Status:** A full GPU run completed on Colab (T4, fp16 QLoRA) — run_id `qwen25-lora-20260810-052502`, logged in `04_docs/training_run_log.md`. That run used the 69-example synthetic `sample_data/` smoke set (`lm-legal-smoke-v1`), so it proves the notebook works end-to-end but is **not** a production candidate — do not promote it.

This version is pointed at `lm-legal-v0.1` (real Sri Lankan statutes). After training, evaluate twice: `test.jsonl` reports **in_corpus_accuracy (chapter-held-out)**; `test_strict.jsonl` reports **accuracy (document-held-out)**. Those two numbers are not interchangeable. Do not promote until both are logged in `version_registry.csv`.

## 0 — Install dependencies (Colab)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # SAFE INSTALL for Colab.
    # We deliberately do NOT touch torch / torchvision / pillow / numpy -- Colab's
    # preinstalled versions of those four are already matched to its GPU driver and
    # CUDA build. Reinstalling or upgrading them is what caused, in order, while
    # building this notebook:
    #   ValueError: numpy.dtype size changed, may indicate binary incompatibility
    #   RuntimeError: operator torchvision::nms does not exist
    #   ImportError: cannot import name '_Ink' from 'PIL._typing'
    # Only the Hugging Face training stack is installed/upgraded here.
    %pip install -q -U \
        transformers \
        accelerate \
        peft \
        bitsandbytes \
        trl \
        datasets \
        sentencepiece \
        einops
    print("Installed Hugging Face training stack (transformers/accelerate/peft/bitsandbytes/trl/datasets).")
    print("NEXT STEP (required): Runtime -> Restart session, then run cell '0b - Verify environment'.")
    print("Do not re-run this install cell in the same session.")
else:
    print("Not running in Colab -- install matching versions from requirements.txt locally.")

print("IN_COLAB =", IN_COLAB)

## 0b — Verify environment (run AFTER "Runtime → Restart session")

Do **not** re-run the install cell above after this. If any line below prints `IMPORT FAILED`, see the troubleshooting note printed at the bottom of this cell's output.

In [ ]:
import importlib


def _v(mod: str) -> str:
    try:
        m = importlib.import_module(mod)
        return getattr(m, "__version__", "unknown")
    except Exception as e:  # noqa: BLE001
        return f"IMPORT FAILED: {type(e).__name__}: {e}"


import torch

print("torch          :", torch.__version__)
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            :", torch.cuda.get_device_name(0))
    print("capability     :", torch.cuda.get_device_capability(0))

for _mod in ["transformers", "accelerate", "peft", "bitsandbytes", "trl", "datasets"]:
    print(f"{_mod:14s}:", _v(_mod))

print(
    "\nIf anything above says IMPORT FAILED: Runtime -> Disconnect and delete runtime, "
    "reconnect on a FRESH GPU runtime, run ONLY the install cell once, Restart session, "
    "then re-run this cell. Do not run the install cell twice in the same runtime."
)

## 1 — CONFIG (single source of truth)

Change hyperparameters **only here**. The run-record cell reads this dict so every saved adapter is reproducible.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

CONFIG = {
    # --- Identity ---
    "run_id": f"qwen25-lora-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
    "project": "LearnMateAI",
    "track": "model-Thevindu",

    # --- Base model ---
    "base_model_id": "Qwen/Qwen2.5-1.5B-Instruct",  # upgrade to 3B/7B only with GPU budget
    "torch_dtype": "bfloat16",  # fallback to float16 on older GPUs
    "use_qlora": True,          # 4-bit; set False for full LoRA in bf16 if VRAM allows

    # --- Dataset (Stage 3 output) ---
    # Real corpus: 01_dataset_pipeline/processed_v01/ (1590 train / 339 val).
    # Not sample_data/ (smoke-v1) and not processed/ (smoke-v2, also synthetic).
    "dataset_version": "lm-legal-v0.1",
    "train_path": "data/train.jsonl",  # Colab: upload processed_v01/train.jsonl and val.jsonl
    "val_path": "data/val.jsonl",
    # VRAM budget cap, not a target. Attention/activation memory and step time both
    # scale with sequence length, so the auto-tune cell measures real token lengths
    # and lowers this to the smallest bucket that still fits the longest example.
    "max_seq_length": 512,
    "auto_seq_length": True,
    "packing": False,

    # --- LoRA / PEFT ---
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "bias": "none",
        "task_type": "CAUSAL_LM",
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    },

    # --- Training ---
    "training": {
        # 1590 training examples, effective batch 8 (4 x 2) => ~199 steps/epoch.
        # 3 epochs => ~597 optimizer steps, about 45-90 min on a free T4. Six epochs
        # was sized for the 46-example smoke set; on this corpus it roughly doubles
        # wall time and raises overfitting risk before the first real eval.
        "num_train_epochs": 3,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 2,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        # eval/save/logging cadence is recomputed from the real step count by the
        # auto-tune cell; the values here are only fallbacks if auto_schedule is off.
        "auto_schedule": True,
        "logging_steps": 5,
        "eval_strategy": "steps",
        "eval_steps": 25,
        "save_strategy": "steps",
        "save_steps": 25,
        "save_total_limit": 2,
        "fp16": False,
        "bf16": True,
        "optim": "paged_adamw_8bit",

        # --- T4 (16 GB) memory guards ---
        # Trades ~20-30% step time for a large activation-memory saving; this is what
        # keeps headroom for a bigger corpus later without re-tuning everything.
        "gradient_checkpointing": True,
        # Qwen2.5's vocab is ~152k, so a batch of logits is enormous. Without this the
        # eval loop gathers full logit tensors and is the most likely OOM in the run --
        # we only need eval_loss (metric_for_best_model), never the predictions.
        "prediction_loss_only": True,
        "eval_accumulation_steps": 1,
        # The collator already pads to the longest sequence *per batch*, so
        # group_by_length buys little here and has broken on some trl versions that
        # drop the length column. Left off deliberately.
        "group_by_length": False,
        "dataloader_num_workers": 2,
        "dataloader_pin_memory": True,
        "report_to": "none",
        "seed": 42,
    },

    # --- Outputs ---
    "output_dir": "adapters",
    "run_records_dir": "run_records",
}

ADAPTER_DIR = Path(CONFIG["output_dir"]) / CONFIG["run_id"]
RUN_RECORD_PATH = Path(CONFIG["run_records_dir"]) / f"{CONFIG['run_id']}.json"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
Path(CONFIG["run_records_dir"]).mkdir(parents=True, exist_ok=True)

print("run_id       :", CONFIG["run_id"])
print("base_model   :", CONFIG["base_model_id"])
print("dataset      :", CONFIG["dataset_version"])
print("adapter_dir  :", ADAPTER_DIR)
print("run_record   :", RUN_RECORD_PATH)

## 2 — Load Stage 3 JSONL and format for chat SFT

In [ ]:
import json
from pathlib import Path
from datasets import Dataset


def _ensure_dataset_files() -> None:
    """If train/val JSONL are missing, offer a Colab upload widget instead of failing."""
    train_p = Path(CONFIG["train_path"])
    val_p = Path(CONFIG["val_path"])
    if train_p.exists() and val_p.exists():
        return

    print("Dataset files not found:")
    print(" ", train_p, "exists=", train_p.exists())
    print(" ", val_p, "exists=", val_p.exists())

    if IN_COLAB:
        print(
            "\nUpload dialog opening -- select train.jsonl and val.jsonl from "
            "model-Thevindu/01_dataset_pipeline/processed_v01/ (lm-legal-v0.1). "
            "Do NOT use sample_data/ or processed/ -- those are the synthetic smoke sets."
        )
        train_p.parent.mkdir(parents=True, exist_ok=True)
        from google.colab import files

        uploaded = files.upload()
        for name, content in uploaded.items():
            dest = train_p.parent / name
            with open(dest, "wb") as f:
                f.write(content)
            print("saved", dest)

        if not (train_p.exists() and val_p.exists()):
            raise FileNotFoundError(
                f"Still missing after upload: {train_p} and/or {val_p}. "
                "Filenames must match CONFIG['train_path'] / CONFIG['val_path']."
            )
    else:
        raise FileNotFoundError(
            f"Missing dataset files: {train_p} / {val_p}. "
            "Copy model-Thevindu/01_dataset_pipeline/processed_v01/{train,val}.jsonl to "
            f"{train_p.parent}/ next to this notebook."
        )


_ensure_dataset_files()


def load_jsonl(path: str):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(CONFIG["train_path"])
val_rows = load_jsonl(CONFIG["val_path"])

assert train_rows, f"Empty train set: {CONFIG['train_path']}"
assert all("messages" in r for r in train_rows), "Expected chat `messages` field from Stage 3"

# Confirm dataset_version lineage
versions = {r.get("dataset_version") for r in train_rows}
print(f"train={len(train_rows)}  val={len(val_rows)}  dataset_versions={versions}")
if CONFIG["dataset_version"] not in versions:
    print("WARNING: CONFIG dataset_version does not match records — update CONFIG before a real run.")

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
train_ds[0]["messages"][:2]

## 3 — Tokenizer + base model (QLoRA or LoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for this notebook. Runtime -> Change runtime type -> GPU (T4).")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["base_model_id"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# T4 (compute capability 7.5) has no real bf16 tensor cores -- use fp16 there.
# Newer GPUs (A100/L4/H100, capability >= 8) use bf16.
_cap_major = torch.cuda.get_device_capability(0)[0]
dtype = torch.bfloat16 if _cap_major >= 8 else torch.float16
CONFIG["torch_dtype_actual"] = str(dtype)
print(f"GPU capability {torch.cuda.get_device_capability(0)} -> using dtype {dtype}")


TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"total VRAM: {TOTAL_VRAM_GB:.1f} GB")


def _load_model(use_qlora: bool):
    quant_cfg = None
    if use_qlora:
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
    kwargs = dict(
        quantization_config=quant_cfg,
        # Pin every module to GPU 0 rather than letting accelerate decide. On a T4
        # "auto" can silently offload layers to CPU when VRAM looks tight, which
        # doesn't error -- it just makes training ~10x slower for no obvious reason.
        # A 1.5B model in 4-bit is ~1.1 GB and comfortably fits, so fail loudly instead.
        device_map={"": 0},
        low_cpu_mem_usage=True,
        # Always pass torch_dtype explicitly (never None). With QLoRA, this only
        # governs the non-quantized modules -- but leaving it None makes
        # from_pretrained fall back to the checkpoint's own declared dtype
        # (Qwen ships as bfloat16), which then leaks into the LoRA adapter layers
        # built on top. That desyncs them from an fp16 GradScaler on GPUs without
        # bf16 support (e.g. T4) and raises:
        #   NotImplementedError: ..._unscale_cuda not implemented for 'BFloat16'
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    # Memory-efficient attention. FlashAttention-2 needs Ampere or newer, so on a T4
    # the best available option is PyTorch SDPA -- still far cheaper than the eager
    # path, which materialises the full attention matrix. Older transformers builds
    # don't accept this kwarg at all, hence the fallback.
    try:
        return AutoModelForCausalLM.from_pretrained(
            CONFIG["base_model_id"], attn_implementation="sdpa", **kwargs
        )
    except (TypeError, ValueError) as exc:
        print(f"sdpa attention unavailable ({type(exc).__name__}); using default attention.")
        return AutoModelForCausalLM.from_pretrained(CONFIG["base_model_id"], **kwargs)


try:
    model = _load_model(CONFIG["use_qlora"])
except Exception as exc:  # noqa: BLE001 -- fall back rather than hard-fail the whole run
    print(f"QLoRA (4-bit) load failed: {type(exc).__name__}: {exc}")
    print("Falling back to non-quantized LoRA load (uses more VRAM).")
    CONFIG["use_qlora"] = False
    model = _load_model(False)

# Incompatible with gradient checkpointing, and useless during training anyway.
model.config.use_cache = False
print("loaded", CONFIG["base_model_id"], "qlora=" + str(CONFIG["use_qlora"]))
print(f"attention impl : {getattr(model.config, '_attn_implementation', 'unknown')}")
print(
    f"VRAM after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB "
    f"of {TOTAL_VRAM_GB:.1f} GB"
)

## 4 — Attach LoRA adapters (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# use_reentrant=False is the non-deprecated checkpointing path (PyTorch 2.9 will
# raise if the flag is left unset) and it is the variant that actually works with
# frozen 4-bit base weights -- the reentrant one can drop grads for the LoRA layers
# because no input to the checkpointed block requires grad.
_gc_kwargs = {"use_reentrant": False}

if CONFIG["use_qlora"]:
    try:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs=_gc_kwargs,
        )
    except TypeError:  # older peft without gradient_checkpointing_kwargs
        model = prepare_model_for_kbit_training(model)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=_gc_kwargs)
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=_gc_kwargs)

# With checkpointing on, the block inputs must require grad or the LoRA layers get
# no gradient at all and the run silently trains nothing.
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_cfg = LoraConfig(**CONFIG["lora"])
model = get_peft_model(model, lora_cfg)

# Force every trainable (LoRA) param to a SAFE dtype for the optimizer/scaler:
#   - bf16 GPUs (Ampere+): trainable params can stay in bfloat16 directly (no
#     GradScaler is used for bf16, so there's no dtype restriction).
#   - fp16 GPUs (e.g. T4): trainable params MUST be float32, not float16.
#     PyTorch's GradScaler (used for fp16 AMP) explicitly refuses to unscale
#     native float16 gradients:  ValueError: "Attempting to unscale FP16
#     gradients." The fp16 speed-up still comes from autocast on the forward/
#     backward math; the actual params/grads the optimizer touches must stay fp32.
# (Leaving adapter weights in whatever the checkpoint happened to declare has
# also caused: NotImplementedError: ..._unscale_cuda not implemented for 'BFloat16'.)
_lora_param_dtype = torch.bfloat16 if dtype == torch.bfloat16 else torch.float32
_n_recast = 0
for _name, _param in model.named_parameters():
    if _param.requires_grad and _param.dtype != _lora_param_dtype:
        _param.data = _param.data.to(_lora_param_dtype)
        _n_recast += 1
if _n_recast:
    print(f"Recast {_n_recast} trainable tensor(s) to {_lora_param_dtype}.")

model.print_trainable_parameters()

## 4b — Auto-tune sequence length and schedule for this GPU + dataset

Measures the **actual** tokenized length of every example and the **actual** optimizer-step count, then sizes `max_seq_length` and the eval/save cadence to match. Both were previously hardcoded for a dataset that no longer exists: a 1024-token cap on data that never exceeds ~400 tokens wastes T4 VRAM, and evaluating every 25 steps never fires on a ~36-step run.

In [ ]:
import math

_t = CONFIG["training"]

# --- 1. Sequence length -------------------------------------------------
# Tokenize through the same chat template training will use, so these are the
# real lengths the collator sees rather than a chars/token guess.
def _rendered_token_len(example) -> int:
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


_lengths = sorted(_rendered_token_len(r) for r in list(train_ds) + list(val_ds))
_p50 = _lengths[len(_lengths) // 2]
_p95 = _lengths[min(len(_lengths) - 1, int(len(_lengths) * 0.95))]
_pmax = _lengths[-1]
print(f"token lengths (n={len(_lengths)}): p50={_p50}  p95={_p95}  max={_pmax}")

_cap = CONFIG["max_seq_length"]
if CONFIG.get("auto_seq_length", False):
    # Round the longest example up to the next power-of-two bucket, clamped to the
    # configured cap (the VRAM budget) and floored at 256 so a tiny sample doesn't
    # produce a cap that a later, longer dataset would silently truncate against.
    _fitted = max(256, 1 << max(0, _pmax - 1).bit_length())
    TUNED_SEQ_LEN = min(_cap, _fitted)
else:
    TUNED_SEQ_LEN = _cap

_n_truncated = sum(1 for x in _lengths if x > TUNED_SEQ_LEN)
if _n_truncated:
    print(
        f"WARNING: {_n_truncated} example(s) exceed {TUNED_SEQ_LEN} tokens and will be "
        f"truncated (longest={_pmax}). Raise CONFIG['max_seq_length'] if that loses answers."
    )

CONFIG["max_seq_length"] = TUNED_SEQ_LEN
print(f"max_seq_length: cap={_cap} -> using {TUNED_SEQ_LEN}")

# --- 2. Step schedule ---------------------------------------------------
_eff_batch = max(
    1, _t["per_device_train_batch_size"] * _t["gradient_accumulation_steps"]
)
STEPS_PER_EPOCH = max(1, math.ceil(len(train_ds) / _eff_batch))
TOTAL_STEPS = max(1, STEPS_PER_EPOCH * int(_t["num_train_epochs"]))
print(
    f"effective batch={_eff_batch}  steps/epoch={STEPS_PER_EPOCH}  "
    f"total optimizer steps={TOTAL_STEPS}"
)

if _t.get("auto_schedule", False):
    # Aim for ~4 evaluations across the run, and never less than one per epoch.
    # save_steps must equal eval_steps or load_best_model_at_end has no checkpoint
    # matching the best eval score to restore.
    _every = max(1, min(STEPS_PER_EPOCH, TOTAL_STEPS // 4))
    _t["eval_steps"] = _every
    _t["save_steps"] = _every
    _t["logging_steps"] = max(1, _every // 2)
    print(
        f"auto schedule -> eval+save every {_every} step(s), "
        f"logging every {_t['logging_steps']} ({TOTAL_STEPS // _every} evals)"
    )

if TOTAL_STEPS < 20:
    print(
        f"WARNING: only {TOTAL_STEPS} optimizer steps -- too few to learn much. "
        "Raise num_train_epochs or lower gradient_accumulation_steps."
    )

## 5 — Train with TRL SFTTrainer

In [ ]:
import inspect
import math

from transformers import TrainingArguments
from trl import SFTTrainer

try:
    from trl import SFTConfig
    _HAS_SFTCONFIG = True
except ImportError:
    _HAS_SFTCONFIG = False


def formatting_func(example):
    # Qwen chat template applied by tokenizer
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )


t = CONFIG["training"]
# Older GPUs may not support bf16
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
fp16 = not bf16_ok
bf16 = bf16_ok and t["bf16"]

base_args = dict(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    num_train_epochs=t["num_train_epochs"],
    per_device_train_batch_size=t["per_device_train_batch_size"],
    per_device_eval_batch_size=t["per_device_eval_batch_size"],
    gradient_accumulation_steps=t["gradient_accumulation_steps"],
    learning_rate=t["learning_rate"],
    lr_scheduler_type=t["lr_scheduler_type"],
    weight_decay=t["weight_decay"],
    max_grad_norm=t["max_grad_norm"],
    logging_steps=t["logging_steps"],
    logging_first_step=True,
    save_strategy=t["save_strategy"],
    save_steps=t["save_steps"],
    save_total_limit=t["save_total_limit"],
    fp16=fp16,
    bf16=bf16,
    optim=t["optim"],
    report_to=t["report_to"],
    seed=t["seed"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # --- T4 memory guards (see CONFIG['training'] for the reasoning) ---
    gradient_checkpointing=t["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    prediction_loss_only=t["prediction_loss_only"],
    eval_accumulation_steps=t["eval_accumulation_steps"],
    group_by_length=t["group_by_length"],
    dataloader_num_workers=t["dataloader_num_workers"],
    dataloader_pin_memory=t["dataloader_pin_memory"],
)

# Ask TrainingArguments what it actually accepts instead of probing with
# try/except TypeError. Probing hides *which* kwarg was rejected, so an unrelated
# removed argument (e.g. warmup_ratio) gets misreported as an eval-strategy problem
# and then re-raised from the fallback line.
def _accepted_kwargs(cls) -> set:
    params = inspect.signature(cls.__init__).parameters
    if any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values()):
        return set()  # accepts **kwargs: don't filter anything out
    return set(params)


def _fit_kwargs(cls, kwargs: dict, label: str) -> dict:
    accepted = _accepted_kwargs(cls)
    if not accepted:
        return dict(kwargs)
    kept = {k: v for k, v in kwargs.items() if k in accepted}
    dropped = sorted(set(kwargs) - set(kept))
    if dropped:
        print(f"WARNING: {label} does not accept {', '.join(dropped)} in this version - dropped.")
    return kept


_TA_PARAMS = _accepted_kwargs(TrainingArguments)


def _apply_schedule_kwargs(target: dict, accepted: set) -> None:
    """Set warmup and eval-strategy under whichever names this version exposes."""
    # warmup_ratio was dropped in favour of warmup_steps in newer transformers.
    if not accepted or "warmup_ratio" in accepted:
        target["warmup_ratio"] = t["warmup_ratio"]
    elif "warmup_steps" in accepted:
        # Prefer the step count the auto-tune cell already derived; only recompute
        # if this cell is run standalone.
        _total_steps = globals().get("TOTAL_STEPS")
        if not _total_steps:
            _eff_batch = max(
                1,
                t["per_device_train_batch_size"] * t["gradient_accumulation_steps"],
            )
            _steps_per_epoch = max(1, math.ceil(len(train_ds) / _eff_batch))
            _total_steps = max(1, _steps_per_epoch * int(t["num_train_epochs"]))
        target["warmup_steps"] = max(1, round(t["warmup_ratio"] * _total_steps))
        print(
            f"warmup_ratio unsupported -> warmup_steps={target['warmup_steps']} "
            f"(of ~{_total_steps} optimizer steps)"
        )

    # transformers renamed evaluation_strategy -> eval_strategy at one point.
    if not accepted or "eval_strategy" in accepted:
        target["eval_strategy"] = t["eval_strategy"]
    elif "evaluation_strategy" in accepted:
        target["evaluation_strategy"] = t["eval_strategy"]
    target["eval_steps"] = t["eval_steps"]


_apply_schedule_kwargs(base_args, _TA_PARAMS)
training_args = TrainingArguments(**_fit_kwargs(TrainingArguments, base_args, "TrainingArguments"))

# SFTTrainer's constructor signature has changed across trl releases (tokenizer= vs
# processing_class=, with/without max_seq_length, TrainingArguments vs SFTConfig).
# Try the combinations in order instead of hardcoding one that might not match
# whatever trl version Colab installed.
trainer = None
_last_err = None
_attempts = [
    {"args": training_args, "tok_kw": "tokenizer", "use_msl": True},
    {"args": training_args, "tok_kw": "processing_class", "use_msl": True},
    {"args": training_args, "tok_kw": "processing_class", "use_msl": False},
    {"args": training_args, "tok_kw": "tokenizer", "use_msl": False},
]

for i, a in enumerate(_attempts, start=1):
    try:
        kwargs = dict(
            model=model,
            args=a["args"],
            train_dataset=train_ds,
            eval_dataset=val_ds,
            formatting_func=formatting_func,
        )
        if a["use_msl"]:
            kwargs["max_seq_length"] = CONFIG["max_seq_length"]
        kwargs[a["tok_kw"]] = tokenizer
        trainer = SFTTrainer(**kwargs)
        print(f"SFTTrainer constructed (attempt {i}: {a['tok_kw']}, max_seq_length={a['use_msl']})")
        break
    except TypeError as e:
        _last_err = e

if trainer is None and _HAS_SFTCONFIG:
    print("Falling back to SFTConfig-based construction (newer trl API)...")
    # SFTConfig subclasses TrainingArguments but its accepted kwargs differ by
    # version, so re-resolve warmup/eval names against SFTConfig itself.
    _SFT_PARAMS = _accepted_kwargs(SFTConfig)
    sft_kwargs = {k: v for k, v in base_args.items()
                  if k not in {"warmup_ratio", "warmup_steps", "eval_strategy",
                               "evaluation_strategy", "eval_steps"}}
    _apply_schedule_kwargs(sft_kwargs, _SFT_PARAMS)
    sft_kwargs["max_seq_length"] = CONFIG["max_seq_length"]
    sft_kwargs["packing"] = CONFIG["packing"]
    sft_args = SFTConfig(**_fit_kwargs(SFTConfig, sft_kwargs, "SFTConfig"))
    for tok_kw in ("processing_class", "tokenizer"):
        try:
            trainer = SFTTrainer(
                model=model,
                args=sft_args,
                train_dataset=train_ds,
                eval_dataset=val_ds,
                formatting_func=formatting_func,
                **{tok_kw: tokenizer},
            )
            print(f"SFTTrainer constructed via SFTConfig ({tok_kw})")
            break
        except TypeError as e:
            _last_err = e

if trainer is None:
    raise RuntimeError(f"Could not construct SFTTrainer with the installed trl version. Last error: {_last_err}")

# Final precision guard, independent of the model-loading/LoRA cells above.
# If this cell is re-run without re-running cells 3-4 first (e.g. after a fix was
# pulled in but the kernel wasn't restarted), `model` may still hold trainable
# (LoRA) params in a dtype that disagrees with the fp16/bf16 decision made just
# above. Re-check and force-fix it right before training starts, using THIS
# cell's own fp16/bf16 flags as the source of truth.
#
# IMPORTANT: trainable params must NEVER be stored as native float16 here.
# PyTorch's GradScaler (used whenever fp16=True) explicitly rejects float16
# gradients: ValueError: "Attempting to unscale FP16 gradients." -- fp16 speed
# comes from autocast on the forward/backward math; the optimizer-visible
# params/grads must stay float32 under fp16 training. Only bf16 training (no
# scaler involved) can keep trainable params natively in bfloat16.
_train_dtype = torch.bfloat16 if bf16 else torch.float32
_n_recast = 0
for _name, _param in model.named_parameters():
    if _param.requires_grad and _param.dtype != _train_dtype:
        _param.data = _param.data.to(_train_dtype)
        _n_recast += 1
if _n_recast:
    print(f"[precision guard] Recast {_n_recast} trainable tensor(s) to {_train_dtype} (fp16={fp16}, bf16={bf16}).")
else:
    print(f"[precision guard] All trainable tensors already match {_train_dtype}.")

# Reset the counter so PEAK_VRAM_GB measures training only, not leftovers from
# model loading or a previous attempt in the same session.
import gc
import time

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

_t0 = time.time()
train_result = trainer.train()
TRAIN_SECONDS = round(time.time() - _t0, 1)
FINAL_TRAIN_LOSS = float(train_result.training_loss)
print("final train loss:", FINAL_TRAIN_LOSS)

eval_metrics = trainer.evaluate()
FINAL_EVAL_LOSS = float(eval_metrics.get("eval_loss", float("nan")))
print("final eval loss:", FINAL_EVAL_LOSS)

PEAK_VRAM_GB = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
_headroom = TOTAL_VRAM_GB - PEAK_VRAM_GB
print(f"\ntrain wall time : {TRAIN_SECONDS}s")
print(f"peak VRAM       : {PEAK_VRAM_GB} GB of {TOTAL_VRAM_GB:.1f} GB "
      f"({_headroom:.1f} GB headroom)")
if _headroom < 1.5:
    print(
        "WARNING: under 1.5 GB headroom. Before scaling the corpus up, lower "
        "per_device_train_batch_size or max_seq_length -- a longer dataset will OOM here."
    )

## 6 — Save adapter weights

In [ ]:
adapter_save_path = ADAPTER_DIR / "adapter"
trainer.model.save_pretrained(str(adapter_save_path))
tokenizer.save_pretrained(str(adapter_save_path))
print("adapter saved to", adapter_save_path)

## 7 — MANDATORY RUN RECORD

Writes hyperparameters, dataset version, and final loss next to the adapter.   An adapter without a run-record is not eligible for evaluation/promotion.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

# Fail loudly if training metrics were never produced
assert "FINAL_TRAIN_LOSS" in dir() or "FINAL_TRAIN_LOSS" in globals(), (
    "FINAL_TRAIN_LOSS missing — run the training cell before writing a run-record."
)

run_record = {
    "run_id": CONFIG["run_id"],
    "project": CONFIG["project"],
    "track": CONFIG["track"],
    "completed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "base_model_id": CONFIG["base_model_id"],
    "dataset_version": CONFIG["dataset_version"],
    "train_path": CONFIG["train_path"],
    "val_path": CONFIG["val_path"],
    "train_examples": len(train_rows),
    "val_examples": len(val_rows),
    "use_qlora": CONFIG["use_qlora"],
    "torch_dtype_actual": CONFIG.get("torch_dtype_actual", CONFIG["torch_dtype"]),
    "lora": CONFIG["lora"],
    # CONFIG['training'] already reflects the auto-tuned eval/save/logging cadence,
    # and max_seq_length is the tuned value -- so this record replays the run as it
    # actually executed, not as it was first declared.
    "training": CONFIG["training"],
    "max_seq_length": CONFIG["max_seq_length"],
    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "total_vram_gb": round(TOTAL_VRAM_GB, 1),
        "peak_vram_gb": PEAK_VRAM_GB if "PEAK_VRAM_GB" in globals() else None,
        "train_seconds": TRAIN_SECONDS if "TRAIN_SECONDS" in globals() else None,
    },
    "schedule": {
        "steps_per_epoch": STEPS_PER_EPOCH if "STEPS_PER_EPOCH" in globals() else None,
        "total_optimizer_steps": TOTAL_STEPS if "TOTAL_STEPS" in globals() else None,
    },
    "final_train_loss": FINAL_TRAIN_LOSS,
    "final_eval_loss": FINAL_EVAL_LOSS if "FINAL_EVAL_LOSS" in dir() or "FINAL_EVAL_LOSS" in globals() else None,
    "adapter_path": str(adapter_save_path),
    "status": "completed",
    "notes": "LoRA/PEFT only — base weights not modified. App must keep Gemini (or other API) fallback if adapter unavailable.",
}

# Persist beside adapter AND in run_records/
sidecar = Path(adapter_save_path) / "run_record.json"
for path in (RUN_RECORD_PATH, sidecar):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(run_record, f, indent=2)
    print("wrote", path)

run_record

## Next steps

1. Run cell **"8 — Download the adapter"** below to get `adapters/<run_id>/` off Colab (includes `adapter/` + `run_record.json`).
2. Log cost/duration in `04_docs/training_run_log.md`.
3. Evaluate with `03_testing_and_versioning/evaluate_candidate.ipynb` — do **not** promote without passing acceptance thresholds.

## 8 — Download the adapter (Colab)

Zips `adapters/<run_id>/` (LoRA weights + `run_record.json`) and triggers a browser download. Run this after training and the run-record cell above.

In [ ]:
import shutil
import sys

# 1. Define IN_COLAB to fix the NameError
IN_COLAB = 'google.colab' in sys.modules

# (Assuming ADAPTER_DIR is already defined in a previous cell)
zip_base = str(ADAPTER_DIR) 
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(ADAPTER_DIR))
print("Created:", zip_path)

# 2. Trigger the download
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    print("Browser download triggered. If nothing happens, check your browser's pop-up blocker.")
else:
    print(f"Not in Colab - find the zip at: {zip_path}")